In [1]:
import torch
import torch.nn as nn
from torchvision import models
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

import matplotlib.pyplot as plt
import numpy as np
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cpu


In [2]:
model = models.resnet18(weights=None)

model.fc = nn.Linear(model.fc.in_features, 2)

model.load_state_dict(torch.load("../../models/resnet18_ants_bees.pth"))

model = model.to(device)
model.eval()

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_sta

In [7]:
from torchvision import datasets, transforms

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

train_dataset = datasets.ImageFolder(
    "../../datasets/hymenoptera_data/train",
    transform=transform
)

val_dataset = datasets.ImageFolder(
    "../../datasets/hymenoptera_data/val",
    transform=transform
)
train_loader = torch.utils.data.DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True
)
val_loader = torch.utils.data.DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False
)

In [8]:
class_names = train_dataset.classes

print(class_names)

['ants', 'bees']


In [9]:
all_preds = []
all_labels = []
with torch.no_grad():

    for images, labels in val_loader:

        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)

        _, preds = torch.max(outputs, 1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

accuracy = accuracy_score(all_labels, all_preds)

print(f"Accuracy: {accuracy:.4f}")
precision = precision_score(
    all_labels,
    all_preds,
    average="weighted"
)

print(f"Precision: {precision:.4f}")
recall = recall_score(
    all_labels,
    all_preds,
    average="weighted"
)

print(f"Recall: {recall:.4f}")
f1 = f1_score(
    all_labels,
    all_preds,
    average="weighted"
)

print(f"F1 Score: {f1:.4f}")

Accuracy: 0.8824
Precision: 0.8855
Recall: 0.8824
F1 Score: 0.8826


In [10]:
print(classification_report(
    all_labels,
    all_preds,
    target_names=class_names
))
cm = confusion_matrix(all_labels, all_preds)

print(cm)

              precision    recall  f1-score   support

        ants       0.84      0.91      0.88        70
        bees       0.92      0.86      0.89        83

    accuracy                           0.88       153
   macro avg       0.88      0.88      0.88       153
weighted avg       0.89      0.88      0.88       153

[[64  6]
 [12 71]]
